In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchattacks
from torchattacks import FGSM, PGD, MIFGSM 
import pandas as pd
from functions import encode_variable
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F
from pathlib import Path

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchattacks
from torchattacks import FGSM, PGD, MIFGSM 
import pandas as pd
from functions import encode_variable
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F
from pathlib import Path

In [3]:
class RealTabularMIFGSM(torchattacks.MIFGSM):
    def __init__(self, model, eps=0.03, alpha=0.007, steps=10, decay=1.0, mask=None, feature_min = None, feature_max = None):
        super().__init__(model, eps, alpha, steps, decay)
        self.loss = nn.CrossEntropyLoss()  
        self.mask = mask 
        self.feature_min = feature_min
        self.feature_max = feature_max
        
    def forward(self, images, labels):
        """Overrides MIFGSM for tabular data."""
        
        adv_images = images.clone().detach()
        adv_images.requires_grad = True 
        
        momentum = 0
        for _ in range(self.steps):
            outputs = self.model(adv_images)
            loss = self.loss(outputs, labels)

            grad = torch.autograd.grad(loss, adv_images, retain_graph=True, create_graph=True, allow_unused=True)[0]

            if grad is None:
                print("ERROR: Gradient is None! adv_images is not in the graph!")
                break  

            grad = grad / torch.mean(torch.abs(grad), dim=(1,), keepdim=True)
            grad = grad + momentum * self.decay
            momentum = grad

            perturbation = self.alpha * grad.sign()
            if self.mask is not None:
                perturbation = perturbation * self.mask  # Zero out protected features

            adv_images = adv_images + perturbation
            adv_images = torch.clamp(adv_images, min=self.feature_min, max=self.feature_max) 
    
        return adv_images

In [4]:
# Generating, training and testing surrogate model
class SurrogateModel(nn.Module):
    def __init__(self, input_dim):
        super(SurrogateModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128) 
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)  # Binary classification (Normal vs. Anomalous)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def test_surrogate(model_surrogate, X_test, y_test, device):
    model_surrogate.eval()
    X_test, y_test = X_test.to(device), y_test.to(device)

    with torch.no_grad():
        y_pred = model_surrogate(X_test).argmax(axis=1).cpu().numpy()

    accuracy = accuracy_score(y_test.cpu().numpy(), y_pred)
    recall = recall_score(y_test.cpu().numpy(), y_pred)
    f1 = f1_score(y_test.cpu().numpy(), y_pred, average='weighted')

    results = {
        "Accuracy": accuracy,
        "Recall": recall,
        "F1 Score": f1
    }
    print(f"Surrogate Model Results: {results}")
    return results

def train_surrogate(X_train, y_train, device):
    input_dim = X_train.shape[1]
    model_surrogate = SurrogateModel(input_dim).to(device)
    optimizer = optim.Adam(model_surrogate.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

    for epoch in range(8):  # epochs
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model_surrogate(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    print("Surrogate model training complete.")
    return model_surrogate

In [ ]:
def process_data(input_data, device):
    data = input_data.dropna(axis=0)
    print(data)
    #print(data.nunique() > 1)
    #data = data.loc[:, data.nunique() > 1]
    data.columns = [col.strip().lower() for col in data.columns]
    new_cols = ['timestamp' if 'starttimestamp' in col else col for col in data.columns]
    data.columns = new_cols
    try:
        data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
        data = data.sort_values(by='timestamp', ascending=True)
    except:
        print("timestamp fallido")
    features = [col for col in data.columns if col not in ['city', 'timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    if "connectortype" in [col for col in data.columns]:
        features = ['connectortype', 'durationcharge' , 'durationsession' , 'energy' , 'tariff' ,
                'cost' , 'meanpower', 'maxpower']
        data = encode_variable(data, data.columns.get_loc("connectortype"))

    for feature in features:
        data[feature] = pd.to_numeric(data[feature], errors='coerce')

    data.dropna()
    #data = data.sample(frac=1).reset_index(drop=True)
    x_data = data.loc[:, features].select_dtypes(include=[np.number])
    y_data = data.loc[:, ['anomaly']].values.ravel()
    print(y_data)

    size = len(x_data)
    size_init = int(size * 0.8)
    x_train = x_data[:size_init]
    y_train = y_data[:size_init]
    x_test = x_data[size_init:]
    y_test = y_data[size_init:]

    #x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=42, stratify=y_data)

    df = pd.DataFrame(x_test, columns=features)
    df["anomaly"] = y_test
    df = df.sample(frac=1).reset_index(drop=True)
    x_test = df.loc[:, features].values
    y_test = df.loc[:, ['anomaly']].values.ravel()
    x_train = np.array(x_train, dtype=np.float32) 
    x_test = np.array(x_test, dtype=np.float32)
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.long).to(device)

    return data, x_train, x_test, y_train, y_test, features
			
def load_anomalous_test_samples(x_test, y_test):
    anomalous_indices = np.where(y_test == 1)[0] 
    X_anomalous = x_test[anomalous_indices]
    y_anomalous = y_test[anomalous_indices]
    return X_anomalous, y_anomalous, anomalous_indices

def process_anomalous_samples(x_train, y_train, x_test, y_test, device):
    # Prepare anomalous x y for perturbation
    X_anomalous_test, y_anomalous_test, anomalous_indices_test = load_anomalous_test_samples(x_test, y_test)
    X_anomalous_train, y_anomalous_train, anomalous_indices_train = load_anomalous_test_samples(x_train, y_train)
    x_anom_train = torch.tensor(X_anomalous_train, dtype=torch.float32).clone().detach().requires_grad_(True).to(device)
    x_anom_train = torch.tensor(X_anomalous_train, dtype=torch.float32).to(device)
    x_anom_train.requires_grad = True
    y_anom_train = torch.tensor(y_anomalous_train, dtype=torch.long).to(device)
    y_anom_train = y_anom_train.view(-1).to(device)
    
    x_anom_test = torch.tensor(X_anomalous_test, dtype=torch.float32).clone().detach().requires_grad_(True).to(device)
    x_anom_test = torch.tensor(X_anomalous_test, dtype=torch.float32).to(device)
    x_anom_test.requires_grad = True
    y_anom_test = torch.tensor(y_anomalous_test, dtype=torch.long).to(device)
    y_anom_test = y_anom_test.view(-1).to(device)

    return x_anom_train, y_anom_train, x_anom_test, y_anom_test, anomalous_indices_test, anomalous_indices_train

def extract_column_stats(df):
    stats = {}
    feature_min = []
    feature_max = []
    for column in df.columns:
        stats[column] = {
            'max': df[column].max(),
            'min': df[column].min(),
            'mean': df[column].mean()
        }
        feature_min.append(df[column].min())
        feature_max.append(df[column].max())
    print(stats)
    print(feature_min)
    print(feature_max)
    return stats, feature_min, feature_max

def generateMask(x): 
    mask = []
    for i in range(x.cpu().numpy().shape[1]):  
        col = x.cpu().numpy()[:, i] 
        unique_values = np.unique(col)  
        if len(unique_values) < 5 or np.issubdtype(col.dtype, np.integer):
            mask.append(0) 
        else:
            mask.append(1)
    print(mask)
    return mask

def save_samples_csv(adversarial_samples, adv_dir, data, features, y):
    adversarial_dataset = pd.DataFrame(adversarial_samples.detach().cpu().numpy(), columns=data.loc[:, features].columns)
    adversarial_dataset['anomaly'] = y.detach().cpu().numpy()
    adversarial_dataset.to_csv(adv_dir, index=False)
    #print(f"{adv_dir} converted to CSV")

def performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha):
    mifgsm = RealTabularMIFGSM(model_surrogate, eps=eps, alpha=alpha, steps=steps, decay=0.8, mask=mask, feature_min=feature_min,
                                    feature_max=feature_max)
    mifgsm_samples = mifgsm(x, y)
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    Path(adv_dir).parent.mkdir(parents=True, exist_ok=True)
    save_samples_csv(mifgsm_samples, adv_dir, data, features, y)

def attack(x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features):
    steps = 20
    epsilons = [0.0]
     
    for value in np.arange(0.003, 0.031, 0.006):
        eps = round(value, 3)
        alpha = round(eps/steps, 4)
        performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)
        epsilons.append(eps)
    for value in np.arange(0.03, 0.3, 0.06):
        eps = round(value, 3)
        alpha = round(eps/steps, 4)
        performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)
        epsilons.append(eps)
    if "train" in stage:
        for value in np.arange(0.003, 0.3, 0.05):
            eps = round(value, 3)
            alpha = round(eps/steps, 4)
            performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)

    return epsilons


In [7]:
def generate_ae_examples():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    data_path = Path('./data')
    original_results = {}
    files = [f for f in data_path.iterdir() if f.is_file()]
    for file in files:
        file_name = file.with_suffix('').as_posix().split('/')[1]
        original_results[file_name] = {}
        print(f"Processing file {file_name}")
        if 'xlsx' in file.name:
            data = pd.read_excel(file)
        else:
            data = pd.read_csv(file)
        data, x_train, x_test, y_train, y_test, features = process_data(data, device)
        print(np.unique(y_train.cpu().numpy(), return_counts=True))
        print(np.unique(y_test.cpu().numpy(), return_counts=True))
        # Train and test surrogate model
        model_surrogate = train_surrogate(x_train, y_train, device)
        original_results[file_name]['results'] = test_surrogate(model_surrogate, x_test, y_test, device)
        original_results[file_name]['x_test'] = x_test
        original_results[file_name]['y_test'] = y_test
        original_results[file_name]['x_train'] = x_train
        original_results[file_name]['y_train'] = y_train
        original_results[file_name]['features'] = features
        original_results[file_name]['model_surrogate'] = model_surrogate 
        # Generate data constraints
        stats, feature_min, feature_max = extract_column_stats(data.loc[:, features])
        mask = generateMask(x_train)
        feature_min = torch.tensor(np.array(feature_min), dtype=torch.float32).to(device)
        feature_max = torch.tensor(np.array(feature_max), dtype=torch.float32).to(device)  
        mask = torch.tensor(np.array(mask), dtype=torch.float32).to(device) 

        x_anom_train, y_anom_train, x_anom_test, y_anom_test, anomalous_indices_test, anomalous_indices_train = process_anomalous_samples(x_train.cpu(), y_train.cpu(), x_test.cpu(), y_test.cpu(), device)
        original_results[file_name]['anomalous_indices_test'] = anomalous_indices_test
        original_results[file_name]['anomalous_indices_train'] = anomalous_indices_train
        epsilons = attack(x_anom_train, y_anom_train, mask, feature_min, feature_max, model_surrogate, file_name, 'train', data, features)
        epsilons = attack(x_anom_test, y_anom_test, mask, feature_min, feature_max, model_surrogate, file_name, 'test', data, features)

    return original_results, epsilons, device
        
original_results, epsilons, device = generate_ae_examples()

Device: cuda
Processing file US_Alabama_clean_WithAnomalies
            Timestamp         P         Q  anomaly
0       4/10/19 21:36  2.497251  0.653574      1.0
1       3/10/19 14:31  5.054901  2.872479      1.0
2       2/10/19 20:36  4.524378  2.513386      1.0
3       3/10/19 16:57  2.743485  1.507386      1.0
4        3/10/19 6:41 -1.512417  0.163565      1.0
...               ...       ...       ...      ...
349995   5/10/19 1:13  0.151000  0.267000      0.0
349996   5/10/19 1:13  0.151000  0.260000      0.0
349997   5/10/19 1:13  0.150000  0.266000      0.0
349998   5/10/19 1:13  0.152000  0.263000      0.0
349999   5/10/19 1:13  0.151000  0.262000      0.0

[350000 rows x 4 columns]
[0. 0. 0. ... 1. 0. 0.]
(array([0, 1]), array([223970,  56030]))
(array([0, 1]), array([56030, 13970]))
Epoch 1, Loss: 0.2906
Epoch 2, Loss: 0.0481
Epoch 3, Loss: 0.0828
Epoch 4, Loss: 0.1972
Epoch 5, Loss: 0.1164
Epoch 6, Loss: 0.1840
Epoch 7, Loss: 0.1843
Epoch 8, Loss: 0.1527
Surrogate model train

In [9]:
# Test examples
def read_adv_data(advPath):
    adversarial_data = pd.read_csv(advPath)
    features = [col for col in adversarial_data.columns if col not in ['timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    adversarial_x_data = adversarial_data.loc[:, features].values
    adversarial_y_data = adversarial_data.loc[:, ['anomaly']].values.ravel()
    return adversarial_x_data, adversarial_y_data, features
    
def test_adversarial_samples(advPath, x_test, y_test, model_surrogate, anomalous_indices, device):
    adversarial_x_data, adversarial_y_data, _ = read_adv_data(advPath)
    X_modified, Y_modified  = x_test.clone().cpu().numpy(), y_test.clone().cpu().numpy()
    X_modified[anomalous_indices] = adversarial_x_data
    Y_modified[anomalous_indices] = adversarial_y_data

    try:
        y_pred = model_surrogate.predict(X_modified)
    except: 
        model_surrogate.eval()
        tensor_X_modified = torch.tensor(np.array(X_modified, dtype=np.float32), dtype=torch.float32).to(device)
        with torch.no_grad():
            y_pred = model_surrogate(tensor_X_modified).argmax(axis=1).cpu().numpy()

    accuracy = accuracy_score(Y_modified, y_pred)
    recall = recall_score(Y_modified, y_pred)
    f1 = f1_score(Y_modified, y_pred)
    return  accuracy, f1, recall

def initializeResults(original_results, file_name, epsilons):
    metrics = ["Accuracy", "F1 Score", "Recall", "EIR"]
    attacks = ["MIFGSM"]
    multi_index = pd.MultiIndex.from_product([attacks, metrics], names=["Attack", "Metric"])
    df_results = pd.DataFrame(index=multi_index, columns=epsilons)
    df_results.loc[("MIFGSM", "Accuracy"), 0.0] = original_results[file_name]['results']["Accuracy"]
    df_results.loc[("MIFGSM", "F1 Score"), 0.0] = original_results[file_name]['results']["F1 Score"]
    df_results.loc[("MIFGSM", "Recall"), 0.0] = original_results[file_name]['results']["Recall"]
    df_results.loc[("MIFGSM", "EIR"), 0.0] = "-"
    return df_results

def obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device):
    # MIFGSM
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results[file].loc[("MIFGSM", "Accuracy"), eps] = accuracy
    df_results[file].loc[("MIFGSM", "F1 Score"), eps] = f1
    df_results[file].loc[("MIFGSM", "Recall"), eps] = recall
    eir = 1 - (df_results[file].loc[("MIFGSM", "Recall"), eps] / max(original_results[file]['results']["Recall"], 1e-8))
    df_results[file].loc[("MIFGSM", "EIR"), eps] = eir * 100

    return df_results

def completeTest(original_results, epsilons, device):
    df_results = {}
    stage = 'test'
    
    for file in original_results:
        df_results[file] = initializeResults(original_results, file, epsilons)
        x_test, y_test = original_results[file]['x_test'], original_results[file]['y_test']
        anomalous_indices = original_results[file]['anomalous_indices_test']
        model_surrogate = original_results[file]['model_surrogate']
        #for value in np.arange(0.003, 0.031, 0.006): 
        #    eps = round(value, 3)
        #    df_results = obtain_adv_results(df_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)
        
        #for value in np.arange(0.03, 0.3, 0.06):
        for value in np.arange(0.003, 0.031, 0.006): 
            eps = round(value, 3)
            df_results = obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)

        for value in np.arange(0.03, 0.3, 0.06):
            eps = round(value, 3)
            df_results = obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)

    return df_results
df_results = completeTest(original_results, epsilons, device)

In [10]:
import pickle
with open("df_results_gb.pkl", "wb") as f:
    pickle.dump(df_results, f)
with open("original_results_gb.pkl", "wb") as f:
    pickle.dump(original_results, f)

In [ ]:
import pickle
with open("df_results_gb.pkl", "rb") as f:
    df_results = pickle.load(f)
with open("original_results_gb.pkl", "rb") as f:
    original_results = pickle.load(f)

In [11]:
df_results['Dundee_Clean_With_Anomalies_All']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.835855  0.833575  0.831271  0.827352  0.824644  0.821984   
       F1 Score  0.828755   0.75935  0.755203  0.748085   0.74312  0.738203   
       Recall    0.644492  0.638943  0.633337  0.623801  0.617212  0.610739   
       EIR              -  0.860909  1.730786  3.210474  4.232804  5.237198   

                    0.030      0.090      0.150     0.210      0.270  
Attack Metric                                                         
MIFGSM Accuracy  0.821082   0.794528   0.774102  0.759328   0.749139  
       F1 Score  0.736525   0.685109   0.642618  0.610134   0.586809  
       Recall    0.608542   0.543926    0.49422  0.458271   0.433476  
       EIR       5.577975  15.603982  23.316295  28.89427  32.741458

In [12]:
df_results['Dundee_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.823704  0.823144  0.822557  0.821103  0.819705  0.818558   
       F1 Score  0.803105  0.610207  0.608406  0.603925   0.59959  0.596015   
       Recall    0.459759  0.457909  0.455967  0.451156  0.446531  0.442738   
       EIR              -  0.402414   0.82495  1.871227  2.877264  3.702213   

                    0.030      0.090      0.150      0.210     0.270  
Attack Metric                                                         
MIFGSM Accuracy  0.818167   0.802729   0.789921   0.780888  0.775043  
       F1 Score   0.59479   0.544727   0.500399   0.467478  0.445394  
       Recall    0.441443   0.390379   0.348011   0.318131  0.298797  
       EIR       3.983903  15.090543  24.305835  30.804829  35.01006

In [13]:
df_results['Boulder_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy   0.99467   0.99454  0.994151  0.993631  0.993501  0.993241   
       F1 Score  0.994633  0.981659  0.980324  0.978537  0.978089  0.977193   
       Recall    0.965665  0.964807  0.962232  0.958798   0.95794  0.956223   
       EIR              -  0.088889  0.355556  0.711111       0.8  0.977778   

                    0.030     0.090     0.150      0.210      0.270  
Attack Metric                                                        
MIFGSM Accuracy  0.993241  0.988951  0.982192   0.976862   0.972572  
       F1 Score  0.977193  0.962172  0.937585   0.917363   0.900519  
       Recall    0.956223  0.927897  0.883262   0.848069   0.819742  
       EIR       0.977778  3.911111  8.533333  12.177778  15.111111

In [14]:
df_results['PaloAlto_2018_2022_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.978597  0.978176  0.977946  0.977602  0.977104  0.976721   
       F1 Score  0.978218  0.926147  0.925311  0.924056  0.922237  0.920833   
       Recall    0.889578  0.886849   0.88536  0.883127  0.879901  0.877419   
       EIR              -  0.306834  0.474198  0.725244  1.087866  1.366806   

                    0.030     0.090      0.150     0.210     0.270  
Attack Metric                                                       
MIFGSM Accuracy   0.97653  0.970518   0.963588  0.956697  0.950379  
       F1 Score   0.92013  0.897579   0.870383  0.841973  0.814645  
       Recall    0.876179  0.837221   0.792308  0.747643    0.7067  
       EIR       1.506276  5.885635  10.934449  15.95537  20.55788

In [15]:
df_results['Perth_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.988785  0.988148  0.987893  0.986491   0.98579  0.984707   
       F1 Score  0.988725  0.961838  0.960986  0.956271  0.953897  0.950207   
       Recall    0.951496  0.947454  0.945837  0.936944  0.932498  0.925627   
       EIR              -  0.424809  0.594732  1.529312  1.996602  2.718777   

                    0.030     0.090     0.150      0.210      0.270  
Attack Metric                                                        
MIFGSM Accuracy  0.984388  0.975658  0.967183   0.958963   0.952909  
       F1 Score  0.949117  0.918341  0.886689   0.854167   0.828975  
       Recall    0.923605   0.86823   0.81447   0.762328   0.723929  
       EIR       2.931181  8.751062  14.40102  19.881054  23.916737

In [16]:
df_results['Netherlands_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.892886  0.892886  0.892886  0.891673  0.890057  0.889248   
       F1 Score  0.879279  0.665826  0.665826  0.660759  0.653944   0.65051   
       Recall    0.506718  0.506718  0.506718   0.50096  0.493282  0.489443   
       EIR              -       0.0       0.0  1.136364  2.651515  3.409091   

                    0.030     0.090      0.150      0.210      0.270  
Attack Metric                                                         
MIFGSM Accuracy  0.888844  0.883994    0.87793   0.876718   0.874293  
       F1 Score  0.648787  0.627756   0.600529   0.594954   0.583668  
       Recall    0.487524  0.464491   0.435701   0.429942   0.418426  
       EIR       3.787879  8.333333  14.015152  15.151515  17.424242

In [17]:
df_results['Canada1_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.956471  0.956429  0.956357    0.9563  0.956271  0.956257   
       F1 Score   0.95478  0.879921  0.879701  0.879524  0.879436  0.879392   
       Recall    0.800143  0.799928  0.799571  0.799284  0.799141  0.799069   
       EIR              -  0.026838  0.071569  0.107354  0.125246  0.134192   

                    0.030     0.090     0.150     0.210     0.270  
Attack Metric                                                      
MIFGSM Accuracy  0.956257  0.955886  0.955557    0.9551  0.954671  
       F1 Score  0.879392  0.878243  0.877225  0.875805  0.874471  
       Recall    0.799069  0.797208  0.795562  0.793271  0.791124  
       EIR       0.134192  0.366792  0.572553   0.85883  1.127214

In [62]:
df_results['Germany_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.893157  0.892243  0.891471  0.889986  0.888714  0.887543   
       F1 Score  0.886156  0.684591  0.681614  0.675843  0.670864  0.666243   
       Recall    0.590551   0.58597  0.582105   0.57466  0.568289  0.562419   
       EIR              -  0.775758  1.430303  2.690909  3.769697  4.763636   

                    0.030      0.090      0.150      0.210      0.270  
Attack Metric                                                          
MIFGSM Accuracy  0.887214      0.876   0.865529   0.856057   0.848271  
       F1 Score  0.664941   0.618931    0.57301   0.528763   0.490282  
       Recall    0.560773   0.504581   0.452112   0.404653   0.365641  
       EIR       5.042424  14.557576  23.442424  31.478788  38.084848

In [18]:
df_results['Portugal_clean_WithAnomalies']

0.000      0.003     0.009      0.015      0.021  \
Attack Metric                                                          
MIFGSM Accuracy  0.914814   0.872814  0.844514   0.823943   0.817143   
       F1 Score  0.906554   0.545603  0.382012   0.237942   0.184506   
       Recall    0.593057   0.382606  0.240802   0.137724   0.103651   
       EIR              -  35.485818   59.3965  76.777308  82.522631   

                    0.027     0.030     0.090      0.150      0.210      0.270  
Attack Metric                                                                   
MIFGSM Accuracy  0.819943  0.821286    0.8667   0.870229   0.870343   0.872529  
       F1 Score  0.206897   0.21744  0.513123   0.532042   0.532647   0.544117  
       Recall    0.117681  0.124409  0.351969   0.369649   0.370222   0.381174  
       EIR       80.15691  79.02233  40.65178  37.670489  37.573929  35.727218

In [19]:
df_results['US_Alabama_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.934757  0.934143  0.933371  0.932157  0.931057  0.929857   
       F1 Score   0.93418  0.830888  0.828567  0.824896   0.82155  0.817878   
       Recall    0.813744  0.810666    0.8068  0.800716  0.795204  0.789191   
       EIR              -  0.378255  0.853272  1.600985  2.278325  3.017241   

                    0.030     0.090      0.150      0.210      0.270  
Attack Metric                                                         
MIFGSM Accuracy  0.929371  0.918914     0.9098   0.900957   0.892871  
       F1 Score  0.816386  0.783309   0.752935   0.721979   0.692298  
       Recall    0.786757  0.734359    0.68869   0.644381   0.603865  
       EIR       3.316327  9.755454  15.367699  20.812808  25.791696

In [20]:
df_results['vancover_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.878843  0.878814  0.878814  0.878814  0.878814  0.878814   
       F1 Score  0.860943  0.587944  0.587944  0.587944  0.587944  0.587944   
       Recall    0.433357  0.433214  0.433214  0.433214  0.433214  0.433214   
       EIR              -  0.033036  0.033036  0.033036  0.033036  0.033036   

                    0.030     0.090     0.150     0.210     0.270  
Attack Metric                                                      
MIFGSM Accuracy  0.878814  0.878743    0.8787  0.878657  0.878586  
       F1 Score  0.587944  0.587601  0.587395  0.587189  0.586846  
       Recall    0.433214  0.432856  0.432641  0.432427  0.432069  
       EIR       0.033036  0.115626   0.16518  0.214734  0.297324

In [21]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def trainModelsTransfer(x_train, y_train):
    mask = ~torch.isnan(x_train).any(dim=1)
    x_train = x_train[mask]
    y_train = y_train[mask]
    models = []
    # 1. - Train Catboost   
    print("[CATBOOST]")     
    catboost_model = CatBoostClassifier()
    catboost_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy(), verbose=False)
    models.append(catboost_model)
    # 2 . - Train LGBM        
    print("[LGBM]")    
    lgb_model = LGBMClassifier ()
    lgb_model. fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(lgb_model)
    # 3. - Train MLP
    print("[MLP]")         
    mlp_model = MLPClassifier()
    mlp_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(mlp_model)
    # 4. - Train RF   
    print("[RF]")      
    rf_model = RandomForestClassifier()
    rf_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(rf_model)
    # 5. - Train  XGB   
    print("[XGB]")      
    xgb_model = XGBClassifier()
    xgb_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(xgb_model)
    return models

def initializeResultsTransfer(epsilons, models, x_test, y_test):
    x_test = x_test.cpu().numpy()
    y_test = y_test.cpu().numpy()
    metrics = ["Accuracy", "F1 Score", "Recall", "EIR"]
    attacks = ["MIFGSM"]
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB"]
    multi_index = pd.MultiIndex.from_product([models_names, attacks, metrics], names=["Models", "Attack", "Metric"])
    df_results = pd.DataFrame(index=multi_index, columns=epsilons)
    for idx, model_name in enumerate(models_names):
        y_pred = models[idx].predict(x_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        for attack in attacks:
            df_results.loc[(model_name, attack, "Accuracy"), 0.0] = accuracy
            df_results.loc[(model_name, attack, "F1 Score"), 0.0] = f1
            df_results.loc[(model_name, attack, "Recall"), 0.0] = recall
            df_results.loc[(model_name, attack, "EIR"), 0.0] = "-"      

        print(recall)

    return df_results

def test_adversarial_samples_transfer(adv_dir, x_test, y_test, ads_model, anomalous_indices, device):
    adversarial_x_data, adversarial_y_data, _ = read_adv_data(adv_dir)
    X_modified, Y_modified  = x_test.clone().cpu().numpy(), y_test.clone().cpu().numpy()
    X_modified[anomalous_indices] = adversarial_x_data
    Y_modified[anomalous_indices] = adversarial_y_data

    y_pred =  ads_model.predict(X_modified)

    accuracy = accuracy_score(Y_modified, y_pred)
    recall = recall_score(Y_modified, y_pred)
    f1 = f1_score(Y_modified, y_pred)
    print(recall)
    return  accuracy, f1, recall

def obtain_adv_results_transfer(df_results, original_results,  file, stage,  x_test, y_test, anomalous_indices, eps, ads_model, model_name, device):
    # MIFGSM
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall = test_adversarial_samples_transfer(adv_dir, x_test, y_test, ads_model, anomalous_indices, device)
    df_results.loc[(model_name, "MIFGSM", "Accuracy"), eps] = accuracy
    df_results.loc[(model_name,"MIFGSM", "F1 Score"), eps] = f1
    df_results.loc[(model_name,"MIFGSM", "Recall"), eps] = recall
    eir = 1 - (df_results.loc[(model_name,"MIFGSM", "Recall"), eps] / df_results.loc[(model_name,"MIFGSM", "Recall"), 0.0])
    df_results.loc[(model_name,"MIFGSM", "EIR"), eps] = eir * 100
    print(f'Recall: {recall}')
    return df_results
def testTransferability(original_results, epsilons):
    transferability_results = {}
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB"]
    try_epsilon = [0.0, 0.003, 0.021, 0.090, 0.210]
    # Test normal
    index = 0
    for file in original_results:
        try:
            print(f"processing file {file} number {index}")
            transferability_results[file] = {}
            x_train = original_results[file]['x_train']
            y_train = original_results[file]['y_train']
            x_test = original_results[file]['x_test']
            y_test = original_results[file]['y_test']
            models = trainModelsTransfer(x_train, y_train)
            df_results = initializeResultsTransfer(try_epsilon, models, x_test, y_test)
            transferability_results[file]['models'] = models
            transferability_results[file]['transfer_results'] = df_results
            for idx, model_name in enumerate(models_names):
                for eps in try_epsilon[1:]:
                    transferability_results[file]['transfer_results'] = obtain_adv_results_transfer(transferability_results[file]['transfer_results'] , original_results  , file, 'test',  x_test, y_test, original_results[file]['anomalous_indices_test'], eps, models[idx], model_name, device)
        except Exception as e:
            print(f"File {file} failed because {e}")  
        #if index > 3: 
        #    print(f"Finish with idx {index}")
        #    break
        index = index + 1

# df_results = completeTest(original_results, epsilons, device)
    return transferability_results

In [22]:
transferability_results = testTransferability(original_results, epsilons)

processing file US_Alabama_clean_WithAnomalies number 0
[CATBOOST]
[LGBM]
[LightGBM] [Info] Number of positive: 56030, number of negative: 223970
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000645 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 280000, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200107 -> initscore=-1.385625
[LightGBM] [Info] Start training from score -1.385625
[MLP]
[RF]
[XGB]
0.8668575518969219
0.8470293486041518
0.7933428775948461
0.888117394416607
0.8629205440229062
0.8635647816750179
Recall: 0.8635647816750179
0.8487473156764496
Recall: 0.8487473156764496
0.7894774516821761
Recall: 0.7894774516821761
0.7052254831782391
Recall: 0.7052254831782391
0.8432355046528275
Recall: 0.8432355046528275
0.8258410880458125
Recall: 

In [23]:
import pickle
with open("transferability_results_gb.pkl", "wb") as f:
    pickle.dump(transferability_results, f)

In [ ]:
import pickle
with open("transferability_results_gb.pkl", "rb") as f:
    transferability_results = pickle.load(f)

In [24]:
transferability_results['Dundee_Clean_With_Anomalies']['transfer_results']

0.000     0.003      0.021      0.090      0.210
Models   Attack Metric                                                       
CATBOOST MIFGSM Accuracy  0.886347  0.882823   0.855445   0.828318   0.802813
                F1 Score  0.780087  0.771712   0.702504   0.625785   0.544891
                Recall     0.66679  0.655134    0.56457   0.474838   0.390472
                EIR              -  1.748058  15.330189  28.787458  41.440067
LGBM     MIFGSM Accuracy  0.855557  0.855025   0.843615    0.82197   0.793808
                F1 Score  0.700389  0.698955    0.66738   0.603068    0.50948
                Recall    0.558464  0.556707   0.518964   0.447364   0.354209
                EIR              -  0.314726    7.07305  19.893987  36.574458
MLP      MIFGSM Accuracy  0.824543  0.823844   0.820935   0.811567   0.798674
                F1 Score  0.617765  0.615657   0.606816   0.577555   0.535219
                Recall     0.46901  0.466698   0.457077   0.426087   0.383441
                EIR              -  0.493097   2.544379   9.151874  18.244576
RF       MIFGSM Accuracy  0.835589   0.83008    0.80726   0.790145   0.767017
                F1 Score  0.668583  0.653631   0.587947   0.534318   0.455027
                Recall    0.548566  0.530342   0.454857   0.398242   0.321739
                EIR              -  3.322091  17.082631  27.403035  41.349073
XGB      MIFGSM Accuracy  0.859025  0.868393   0.853375   0.829772   0.802981
                F1 Score  0.710803  0.735112   0.695688   0.628502   0.543333
                Recall     0.57308   0.60407   0.554394   0.476318   0.387697
                EIR              - -5.407587   3.260694  16.884584  32.348668

In [25]:
transferability_results['PaloAlto_2018_2022_Clean_With_Anomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy   0.98189  0.981698   0.98055  0.971667   0.955854
                F1 Score  0.940089  0.939417  0.935369  0.902989   0.840194
                Recall    0.920844  0.919603  0.912159  0.854591   0.752109
                EIR              -  0.134735  0.943142  7.194826  18.323902
LGBM     MIFGSM Accuracy  0.977142   0.97718  0.975764  0.967609    0.95214
                F1 Score  0.923294  0.923433  0.918291  0.887709   0.824684
                Recall    0.891563  0.891811   0.88263  0.829777   0.729529
                EIR              - -0.027832  1.001948  6.930142  18.174228
MLP      MIFGSM Accuracy  0.981086   0.98078  0.979746  0.973581    0.96378
                F1 Score  0.936812  0.935723  0.932031  0.909473   0.871572
                Recall    0.908685    0.9067       0.9   0.86005   0.796526
                EIR              -   0.21846  0.955762  5.352267  12.342982
RF       MIFGSM Accuracy   0.98055   0.98032   0.97875  0.969217   0.954246
                F1 Score  0.935517  0.934705  0.929128  0.893959    0.83382
                Recall    0.914392  0.912903   0.90273  0.840943   0.743921
                EIR              -  0.162822  1.275441  8.032564  18.643148
XGB      MIFGSM Accuracy  0.980014  0.979822  0.978099  0.971131   0.956811
                F1 Score  0.933452  0.932772   0.92661  0.900946   0.844156
                Recall    0.908437  0.907196   0.89603  0.850868   0.758065
                EIR              -  0.136575  1.365747  6.337066  16.552854

In [26]:
transferability_results['Netherlands_Clean_With_Anomalies']['transfer_results']

0.000     0.003      0.021     0.090      0.210
Models   Attack Metric                                                      
CATBOOST MIFGSM Accuracy   0.93654  0.936136   0.942199  0.934115   0.911075
                F1 Score  0.827283  0.825991    0.84507  0.819491   0.739953
                Recall    0.721689   0.71977    0.74856  0.710173   0.600768
                EIR              -  0.265957  -3.723404  1.595745  16.755319
LGBM     MIFGSM Accuracy  0.934519  0.942199   0.940986  0.930073   0.930073
                F1 Score  0.821978  0.846071   0.842333  0.807564   0.807564
                Recall     0.71785  0.754319    0.74856  0.696737   0.696737
                EIR              - -5.080214  -4.278075  2.941176   2.941176
MLP      MIFGSM Accuracy  0.935327  0.934519   0.932094  0.922393   0.906225
                F1 Score  0.834369   0.83195   0.824635  0.794433   0.740492
                Recall    0.773512  0.769674   0.758157  0.712092   0.635317
                EIR              -  0.496278   1.985112  7.940447  17.866005
RF       MIFGSM Accuracy  0.923201  0.924818   0.937753  0.932902   0.920372
                F1 Score  0.785553  0.791011   0.832972  0.817582   0.775882
                Recall    0.667946  0.675624   0.737044  0.714012   0.654511
                EIR              - -1.149425 -10.344828 -6.896552   2.011494
XGB      MIFGSM Accuracy  0.937753  0.950687   0.947454  0.947049   0.927243
                F1 Score  0.833333  0.872385   0.862869  0.861668   0.799555
                Recall    0.738964  0.800384   0.785029  0.783109    0.68906
                EIR              - -8.311688  -6.233766 -5.974026   6.753247

In [27]:
transferability_results['Boulder_Clean_With_Anomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy   0.99506    0.9948   0.99506  0.991941   0.976992
                F1 Score  0.983435  0.982548  0.983435  0.972687   0.917865
                Recall     0.96824  0.966524   0.96824  0.947639   0.848927
                EIR              -  0.177305       0.0   2.12766  12.322695
LGBM     MIFGSM Accuracy  0.992201  0.987651  0.985961  0.983491   0.978292
                F1 Score  0.973568  0.957494  0.951395  0.942351   0.922792
                Recall    0.948498  0.918455  0.907296  0.890987   0.856652
                EIR              -  3.167421  4.343891  6.063348   9.683258
MLP      MIFGSM Accuracy   0.99714   0.99701   0.99532  0.991681   0.981152
                F1 Score  0.990493  0.990056  0.984348  0.971831    0.93382
                Recall    0.983691  0.982833  0.971674  0.947639   0.878112
                EIR              -   0.08726   1.22164  3.664921  10.732984
RF       MIFGSM Accuracy  0.987911  0.987781  0.989471  0.982842   0.973222
                F1 Score  0.958426  0.957961  0.963984  0.939945   0.903013
                Recall    0.920172  0.919313  0.930472  0.886695   0.823176
                EIR              -  0.093284 -1.119403   3.63806  10.541045
XGB      MIFGSM Accuracy  0.993891   0.99519    0.9948  0.991681   0.971923
                F1 Score  0.979431  0.983878  0.982548  0.971781   0.897921
                Recall    0.960515  0.969099  0.966524  0.945923   0.815451
                EIR              - -0.893655 -0.625559  1.519214   15.10277

In [28]:
transferability_results['Perth_Clean_With_Anomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy  0.993628  0.994074   0.99503  0.990187   0.977251
                F1 Score  0.979407  0.980876   0.98401   0.96793   0.922374
                Recall    0.961196  0.964026  0.970089  0.939369   0.857316
                EIR              - -0.294365 -0.925147  2.270816  10.807401
LGBM     MIFGSM Accuracy   0.98528  0.983241  0.981648  0.979099   0.974447
                F1 Score  0.951399  0.944291  0.938671  0.929553   0.912503
                Recall    0.913905   0.90097  0.890865  0.874697    0.84519
                EIR              -  1.415303  2.521008  4.290137   7.518797
MLP      MIFGSM Accuracy  0.991461  0.991397  0.990314  0.984515   0.973491
                F1 Score  0.972337  0.972125  0.968504   0.94868   0.908812
                Recall      0.9519  0.951496  0.944624  0.907842   0.837914
                EIR              -  0.042463  0.764331   4.62845  11.974522
RF       MIFGSM Accuracy  0.985025  0.985471  0.987702   0.98254   0.970751
                F1 Score  0.950222  0.951777  0.959479  0.941478   0.897932
                Recall    0.906629  0.909458  0.923605  0.890865   0.816087
                EIR              - -0.312082 -1.872492  1.738743   9.986625
XGB      MIFGSM Accuracy  0.991907  0.994138  0.994584  0.991206   0.975212
                F1 Score   0.97369  0.981078  0.982543  0.971346   0.914786
                Recall    0.949879  0.964026  0.966855  0.945432   0.843977
                EIR              - -1.489362 -1.787234  0.468085  11.148936

In [29]:
transferability_results['US_Alabama_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy  0.917129  0.916471  0.913514  0.901686   0.884871
                F1 Score  0.806769  0.804937  0.796627  0.762198   0.709722
                Recall    0.866858  0.863565  0.848747  0.789477   0.705225
                EIR              -  0.379851  2.089182  8.926507  18.645747
LGBM     MIFGSM Accuracy  0.921886  0.921129  0.917657  0.906671   0.890186
                F1 Score  0.812316  0.810151  0.800125  0.767252   0.714397
                Recall    0.847029  0.843236  0.825841  0.770795   0.688189
                EIR              -    0.4479  2.501479  9.000254  18.752641
MLP      MIFGSM Accuracy  0.933086  0.932314  0.929714  0.920157   0.903114
                F1 Score  0.825549  0.823183  0.815135  0.784583   0.726002
                Recall    0.793343  0.789477   0.77645  0.728561   0.643164
                EIR              -  0.487233  2.129387  8.165659  18.929893
RF       MIFGSM Accuracy    0.8936  0.892757  0.890357  0.880114   0.864829
                F1 Score   0.76914  0.766885  0.760418  0.732039   0.687186
                Recall    0.888117  0.883894  0.871868  0.820544   0.743951
                EIR              -  0.475538  1.829612  7.608608  16.232772
XGB      MIFGSM Accuracy  0.910857  0.910114  0.907157  0.895429   0.878371
                F1 Score  0.794399   0.79233  0.784022  0.749915   0.696752
                Recall    0.862921  0.859198  0.844381  0.785612   0.700143
                EIR              -  0.431356  2.148486  8.958938  18.863542

In [30]:
transferability_results['Germany_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021     0.090     0.210
Models   Attack Metric                                                    
CATBOOST MIFGSM Accuracy  0.893529  0.893529  0.893529  0.893471  0.893371
                F1 Score  0.704093  0.704093  0.704093  0.703888  0.703527
                Recall    0.634717  0.634717  0.634717  0.634431   0.63393
                EIR              -       0.0       0.0  0.045111  0.124055
LGBM     MIFGSM Accuracy  0.894871  0.894857  0.894857  0.894843  0.894843
                F1 Score  0.702246  0.702193  0.702193  0.702141  0.702141
                Recall    0.621188  0.621117  0.621117  0.621045  0.621045
                EIR              -  0.011523  0.011523  0.023047  0.023047
MLP      MIFGSM Accuracy  0.876229  0.876229  0.876229  0.876229  0.876186
                F1 Score  0.642515  0.642515  0.642515  0.642515  0.642347
                Recall    0.557337  0.557337  0.557337  0.557337  0.557122
                EIR              -       0.0       0.0       0.0  0.038531
RF       MIFGSM Accuracy    0.8645  0.864471  0.864429  0.864357  0.864343
                F1 Score  0.668808  0.668715  0.668576  0.668343  0.668297
                Recall     0.68554  0.685397  0.685183  0.684825  0.684753
                EIR              -  0.020883  0.052208  0.104417  0.114859
XGB      MIFGSM Accuracy  0.893371  0.893371  0.893371    0.8934  0.893314
                F1 Score  0.703574  0.703574  0.703574  0.703677  0.703368
                Recall    0.634073  0.634073  0.634073  0.634216  0.633787
                EIR              -       0.0       0.0 -0.022578  0.045157

In [31]:
transferability_results['Canada1_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021     0.090     0.210
Models   Attack Metric                                                    
CATBOOST MIFGSM Accuracy  0.996443  0.996414  0.996443  0.996386  0.996414
                F1 Score  0.991143  0.991071  0.991143  0.990999  0.991071
                Recall     0.99728  0.997137   0.99728  0.996994  0.997137
                EIR              -  0.014355       0.0  0.028711  0.014355
LGBM     MIFGSM Accuracy  0.997286  0.997257  0.997157  0.997186  0.997186
                F1 Score  0.993201  0.993129  0.992876  0.992948  0.992948
                Recall    0.993343    0.9932  0.992699  0.992842  0.992842
                EIR              -  0.014412  0.064856  0.050443  0.050443
MLP      MIFGSM Accuracy  0.967871  0.967857  0.967814    0.9675    0.9668
                F1 Score  0.918829   0.91879  0.918673  0.917814  0.915895
                Recall    0.911167  0.911095   0.91088  0.909306  0.905798
                EIR              -  0.007856  0.031424  0.204258  0.589206
RF       MIFGSM Accuracy  0.994343    0.9943  0.994329  0.994271  0.994286
                F1 Score  0.985988  0.985881  0.985952  0.985809  0.985845
                Recall    0.997351  0.997137   0.99728  0.996994  0.997065
                EIR              -  0.021532  0.007177  0.035886  0.028709
XGB      MIFGSM Accuracy  0.984557  0.984586  0.984543  0.984514  0.984371
                F1 Score  0.962175  0.962248  0.962139  0.962066  0.961703
                Recall     0.98418  0.984324  0.984109  0.983966   0.98325
                EIR              - -0.014547  0.007273   0.02182  0.094552

In [32]:
transferability_results['vancover_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021     0.090     0.210
Models   Attack Metric                                                    
CATBOOST MIFGSM Accuracy  0.918914  0.918857  0.918829  0.918871  0.918929
                F1 Score  0.786633   0.78645  0.786359  0.786496  0.786678
                Recall    0.748962  0.748676  0.748533  0.748747  0.749034
                EIR              -   0.03823  0.057345  0.028672 -0.009557
LGBM     MIFGSM Accuracy  0.900314  0.900286  0.900271  0.900286  0.899943
                F1 Score  0.722985  0.722884  0.722833  0.722884  0.721666
                Recall    0.651825  0.651682  0.651611  0.651682  0.649964
                EIR              -  0.021964  0.032945  0.021964  0.285526
MLP      MIFGSM Accuracy  0.872829  0.872829  0.872829  0.872814  0.872729
                F1 Score   0.61967   0.61967   0.61967  0.619611  0.619257
                Recall    0.519112  0.519112  0.519112  0.519041  0.518611
                EIR              -       0.0       0.0  0.013789  0.096525
RF       MIFGSM Accuracy  0.916486  0.916486  0.916586  0.916543  0.916471
                F1 Score  0.785231  0.785231  0.785544   0.78541  0.785187
                Recall    0.764996  0.764996  0.765497  0.765283  0.764925
                EIR              -       0.0   -0.0655 -0.037429  0.009357
XGB      MIFGSM Accuracy    0.8815    0.8815  0.881486    0.8815  0.881514
                F1 Score  0.670663  0.670663  0.670611  0.670663  0.670716
                Recall    0.604581  0.604581   0.60451  0.604581  0.604653
                EIR              -       0.0   0.01184       0.0  -0.01184

In [33]:
transferability_results['Portugal_clean_WithAnomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.926786   0.894857   0.858071   0.884171   0.892171
                F1 Score  0.804307   0.692744   0.535292   0.650608   0.682403
                Recall    0.753901   0.593916   0.409592   0.540372   0.580458
                EIR              -  21.221041  45.670338  28.323205  23.006077
LGBM     MIFGSM Accuracy  0.926343   0.892743   0.852171   0.879457   0.886186
                F1 Score  0.798877   0.677547   0.493837   0.622528   0.650953
                Recall    0.732999   0.564639   0.361346   0.498067   0.531782
                EIR              -   22.96875  50.703125  32.050781  27.451172
MLP      MIFGSM Accuracy  0.914557   0.874814   0.810829   0.855657   0.870829
                F1 Score  0.740689   0.567963   0.162111   0.466582    0.54799
                Recall    0.611453   0.412312   0.091696   0.316321   0.392341
                EIR              -  32.568485  85.003512  48.267385  35.834699
RF       MIFGSM Accuracy  0.916286   0.897814   0.870114   0.889386   0.895657
                F1 Score  0.798778   0.742966   0.648822   0.715738   0.736108
                Recall     0.83257   0.740014   0.601217   0.697781   0.729205
                EIR              -  11.116843  27.787808  16.189494  12.415098
XGB      MIFGSM Accuracy  0.926457   0.899329   0.865871     0.8877   0.891543
                F1 Score  0.804229   0.711153   0.574292   0.666667   0.681704
                Recall    0.756908   0.620974   0.453329   0.562706   0.581961
                EIR              -  17.959145  40.107812  25.657273  23.113297